# 01 · Complete preprocessing run (protocol 1.1)

Sandesh Lamichhane, Assistant Professor, Yenepoya (Deemed to be University).

This one notebook does the whole remaining preprocessing. Run every cell **from top to bottom**, one at a time,
waiting for each to finish. You do not need to add, paste or edit anything.

**Before you start:** *Runtime → Change runtime type → High-RAM* (CPU is fine, no GPU needed).

| Step | What it does | Time |
|---|---|---|
| 1 | connect Drive and GitHub, update the code, run the tests | 2 min |
| 2 | schema profile of all 10 datasets | 5 min |
| 3 | CICIDS2017 | 10 min |
| 4 | CICIoMT2024 | 15 min |
| 5 | ToN_IoT | 30–45 min |
| 6 | CIC-IoT2023 | 45–60 min |
| 7 | CSE-CIC-IDS2018, shared CIC features, cross-dataset overlap | 1.5–3 h |
| 8 | LANL host-hour features and Level 3 split | 1–3 h |
| 9 | summary table | 1 min |

Every step saves its results to Drive and pushes them to GitHub before the next one starts, so nothing is lost
if Colab disconnects. **If it disconnects:** run step 1 again, then continue from the step that did not finish.

**Where results go:** reports, data cards and logs in
`MyDrive/PhD-AI-Cyber/code/phd-ai-threat-detection/{results/data_audit, docs/data_cards}` and the same paths on
GitHub; cleaned data in `MyDrive/PhD-AI-Cyber/data/processed/`, splits in `MyDrive/PhD-AI-Cyber/data/splits/`.

## Step 1 · Connect and update (expected end: `in sync with GitHub at …` and `26 passed`)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, pathlib, subprocess
DRIVE_ROOT = '/content/drive/MyDrive/PhD-AI-Cyber'
REPO_DIR   = f'{DRIVE_ROOT}/code/phd-ai-threat-detection'
os.environ['PHD_DRIVE_ROOT'] = DRIVE_ROOT
os.environ['PHD_SCRATCH'] = '/content/scratch'
TOKEN  = userdata.get('GITHUB_TOKEN')
REMOTE = f'https://{TOKEN}@github.com/sandesh20lamichhane/phd-ai-threat-detection.git'

import datetime as _dt, os, pathlib, shutil, subprocess

RECORD_DIRS = ('results/', 'docs/data_cards/', 'config/feature_mapping/')

def git(*args, check=False):
    r = subprocess.run(['git', '-C', REPO_DIR, *args], capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(TOKEN or '\0', '***').strip()
    if check and r.returncode != 0:
        print(out)
        raise SystemExit(f'git {args[0]} failed (see message above)')
    return r.returncode, out

def sync(message=None):
    """Make the Drive copy of the repo identical to GitHub, keeping every result produced here.

    Never merges or rebases, so open notebooks or other local edits cannot block it:
    1. remember result files changed here (results/, docs/data_cards/, config/feature_mapping/)
    2. back up every other local change to MyDrive/PhD-AI-Cyber/backups/sync_<time>/
    3. reset the copy to GitHub's main, put the result files back, commit and push them
    """
    git('config', 'user.name', 'Sandesh Lamichhane')
    git('config', 'user.email', 'san30lamichhane@gmail.com')
    git('rebase', '--abort'); git('merge', '--abort')      # clear any half-finished earlier pull
    for attempt in range(3):
        git('fetch', REMOTE, 'main', check=True)
        base = git('merge-base', 'HEAD', 'FETCH_HEAD')[1] or 'FETCH_HEAD'
        changed = set(git('diff', '--name-only', base)[1].split())
        untracked = set(git('ls-files', '--others', '--exclude-standard')[1].split())
        # untracked non-record files survive the reset unless GitHub has a file at the same path
        untracked = {f for f in untracked if f.startswith(RECORD_DIRS) or git('cat-file', '-e', f'FETCH_HEAD:{f}')[0] == 0}
        changed |= untracked
        keep = sorted(f for f in changed if f.startswith(RECORD_DIRS) and pathlib.Path(REPO_DIR, f).is_file())
        other = sorted(f for f in changed if not f.startswith(RECORD_DIRS) and pathlib.Path(REPO_DIR, f).is_file())
        stamp = _dt.datetime.now(_dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
        hold = pathlib.Path('/content/sync_hold') / stamp if pathlib.Path('/content').exists() else pathlib.Path(REPO_DIR).parent / f'.sync_hold_{stamp}'
        for f in keep:
            (hold / f).parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(pathlib.Path(REPO_DIR, f), hold / f)
        if other:
            bak = pathlib.Path(DRIVE_ROOT, 'backups', f'sync_{stamp}')
            for f in other:
                (bak / f).parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(pathlib.Path(REPO_DIR, f), bak / f)
            print(f'{len(other)} other local change(s) backed up to {bak} (e.g. notebook outputs)')
        git('reset', '--hard', 'FETCH_HEAD', check=True)
        for f in keep:
            dest = pathlib.Path(REPO_DIR, f)
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(hold / f, dest)
        if keep:
            git('add', '--', *keep, check=True)
            if git('diff', '--cached', '--quiet')[0] != 0:
                git('commit', '-m', message or 'Save results produced in Colab', check=True)
        code, out = git('push', REMOTE, 'HEAD:main')
        if code == 0:
            head = git('log', '--oneline', '-1')[1]
            print(f'in sync with GitHub at {head}' + (f'; {len(keep)} result file(s) saved' if keep else ''))
            return
        print('GitHub moved while saving; trying again')
    raise SystemExit('could not save to GitHub after 3 tries: copy the messages above into the project thread')


sync('Save results from an earlier Colab run')      # update the code; results made here are kept
assert pathlib.Path(REPO_DIR, 'src/data/pipeline.py').exists()
%cd $REPO_DIR
!pip -q install -r requirements.txt
!python -m pytest -q tests

def push(message):
    """Save results from the Drive copy of the repo to GitHub, so both hold the same records."""
    sync(message)
    print('saved to GitHub:', message)

def run(*args, message):
    """Run one preprocessing command (its output shows below); push the results only if it succeeded."""
    ip = get_ipython()
    ip.system('python scripts/preprocess.py ' + ' '.join(args))
    ok = ip.user_ns.get('_exit_code', 1) == 0
    push(message if ok else message + ' (FAILED, log only)')
    if not ok:
        raise SystemExit('This step failed. Copy the error above into the project thread.')


## Step 2 · Schema profile (expected: a table with `none` under problems, except LANL's note)

In [ ]:
run('--profile', message='Schema profile (protocol 1.1)')
from IPython.display import Markdown
Markdown(open('results/data_audit/schema_profile.md').read())

## Step 3 · CICIDS2017

In [ ]:
run('--dataset', 'cicids2017_improved', message='Preprocess CICIDS2017 (protocol 1.1)')

## Step 4 · CICIoMT2024

In [ ]:
run('--dataset', 'ciciomt2024', message='Preprocess CICIoMT2024 (protocol 1.1)')

## Step 5 · ToN_IoT

In [ ]:
run('--dataset', 'ton_iot', message='Preprocess ToN_IoT (protocol 1.1)')

## Step 6 · CIC-IoT2023

In [ ]:
run('--dataset', 'ciciot2023', message='Preprocess CIC-IoT2023 (protocol 1.1)')

## Step 7 · CSE-CIC-IDS2018, then freeze the shared CIC feature list and measure cross-dataset overlap

In [ ]:
run('--dataset', 'csecicids2018_improved', message='Preprocess CSE-CIC-IDS2018 (protocol 1.1)')
run('--shared-features', message='Freeze shared CICIDS2017/CSE-CIC-IDS2018 feature list')
run('--cross-overlap', message='Cross-dataset overlap, CICIDS2017 train vs external targets')

## Step 8 · LANL (resumes per source file if Colab disconnects)

In [ ]:
run('--dataset', 'lanl_cyber1', message='LANL host-hour features, compromises and Level 3 split (protocol 1.1)')

## Step 9 · Summary (expected: one row per processed dataset). Then tell the project thread you are done.

In [ ]:
import json, glob
import pandas as pd
rows = []
for f in sorted(glob.glob('results/data_audit/*/report.json')):
    r = json.load(open(f))
    c = r.get('counts', {})
    rows.append({'dataset': r['dataset'], 'rows read': c.get('rows_read'), 'duplicates removed': c.get('exact_duplicates_removed'),
                 'rows kept': c.get('rows_out', r.get('host_hours')), 'built': r['run']['utc'],
                 'code': (r['run']['git_commit'] or '')[:7], 'pilot?': bool(r.get('max_rows'))})
pd.DataFrame(rows)